# CLEAN

## IMPORTS

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

##  DataLoader Class

In [ ]:
class DiabeticDataLoader:
    def __init__(self, df_raw):
        self.df_raw = df_raw.copy()
        self.df_clean = None
        self.df_no_outliers = None
        self.scaler = None
        self.df_treated = None
        

    def clean_data(self):
        df = self.df_raw.copy()
        
        # Drop constant columns
        constant_cols = ['examide', 'citoglipton']
        df.drop(columns=constant_cols, inplace=True, errors='ignore')
        
        # Replace '?' with NaN in weight
        df['weight'] = df['weight'].replace('?', np.nan)
        
        # Handle max_glu_serum and A1Cresult: 'None' means test not performed
        for col in ['max_glu_serum', 'A1Cresult']:
            df[col] = df[col].replace('None', np.nan)
            df[col + '_measured'] = (~df[col].isna()).astype(int)
            
        # Create weight recorded indicator
        df['weight_recorded'] = (~df['weight'].isna()).astype(int)
        
        # Convert weight to categorical (keep original bins)
        df['weight'] = df['weight'].astype('category')
        
        # Convert categorical columns to category dtype
        categorical_cols = [
            'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id', 'payer_code', 'medical_specialty',
            'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult',
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone',
            'change', 'diabetesMed', 'readmitted'
        ]
        
        for col in categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')
        
        self.df_clean = df
        return df
    
    def treat_outliers(self):
        if self.df_clean is None:
            raise ValueError("Data must be cleaned before outlier treatment.")
        
        df = self.df_clean.copy()
        
        # Define capping thresholds and features to treat
        outlier_rules = {
            'time_in_hospital': (1, 14),
            'num_lab_procedures': (0, 85),
            'num_medications': (0, 60),
            'number_outpatient': (0, 20),
            'number_emergency': (0, 10),
            'number_inpatient': (0, 10)
        }
        
        # Apply capping and create outlier flags
        for feature, (lower, upper) in outlier_rules.items():
            if feature in df.columns:
                flag_col = feature + '_outlier_flag'
                df[flag_col] = ((df[feature] < lower) | (df[feature] > upper)).astype(int)
                df[feature] = df[feature].clip(lower=lower, upper=upper)
        
        # Features left unchanged (e.g., num_procedures, number_diagnoses)
        self.df_treated = df
        return df

    def remove_outliers(self):
        if self.df_clean is None:
            raise ValueError("Data must be cleaned before outlier removal.")
        
        df = self.df_clean.copy()
        
        # Define outlier thresholds
        thresholds = {
            'time_in_hospital': (1, 14),
            'num_lab_procedures': (0, 85),
            'num_medications': (0, 60),
            'number_outpatient': (0, 20),
            'number_emergency': (0, 10),
            'number_inpatient': (0, 10)
        }
        
        # Create boolean mask for rows without outliers
        mask = pd.Series(True, index=df.index)
        for feature, (low, high) in thresholds.items():
            if feature in df.columns:
                mask &= df[feature].between(low, high)
                
        # Filter out rows with outliers
        df_no_outliers = df.loc[mask].copy()
        self.df_no_outliers = df_no_outliers
        return df_no_outliers

    def standardize_features(self):
        if self.df_no_outliers is None:
            raise ValueError("Outliers must be removed before standardization.")
            
        df = self.df_no_outliers.copy()
        numeric_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient',
            'num_procedures', 'number_diagnoses'
        ]
        
        # Initialize scaler and fit-transform numeric features
        self.scaler = StandardScaler()
        df[numeric_cols] = self.scaler.fit_transform(df[numeric_cols])
        self.df_no_outliers = df
        return df

    def get_clean_data(self):
        if self.df_clean is None:
            self.clean_data()
        return self.df_clean
    
    def get_outlier_treated_data(self):
        if self.df_treated is None:
            self.treat_outliers()
        return self.df_treated

    def get_no_outliers_data(self):
        if self.df_no_outliers is None:
            self.remove_outliers()
        return self.df_no_outliers

    def get_features_target(self):
        df = self.get_no_outliers_data()
        exclude_cols = ['encounter_id', 'patient_nbr', 'readmitted']
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        X = df[feature_cols]
        y = df['readmitted']
        return X, y

    def get_train_test_split(self, test_size=0.2, random_state=42):
        X, y = self.get_features_target()
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=random_state
        )
        return X_train, X_test, y_train, y_test

### How to Use It

| # | Method                       | Description                                |
|---|------------------------------|--------------------------------------------|
| 1 | `get_clean_data()`           | Full cleaned dataframe                     |
| 1 | `get_outlier_treated_data()` | Flagging  outliers                         |
| 1 | `get_no_outliers_data()`     | Removing outliers                          |
| 1 | `standardize_features()`     | Standardizing features                     |
| 2 | `get_features_target()`      | Modeling features and target (no outliers) |
| 3 | `get_train_test_split()`     | Reproducible encoded train/test split      |

In [4]:
df_diabetic_data = pd.read_csv(os.path.join('..','data', 'raw','diabetic_data.csv'))

In [7]:
df_diabetic_loader = DiabeticDataLoader(df_diabetic_data)
df_diabetic_clean = df_diabetic_loader.get_clean_data()
df_diabetic_treated = df_diabetic_loader.get_outlier_treated_data()
df_diabetic_no_outliers = df_diabetic_loader.get_no_outliers_data()
df_diabetic_no_outliers_standardized = df_diabetic_loader.standardize_features()
df_diabetic_features, df_diabetic_target = df_diabetic_loader.get_features_target()
df_X_train, df_X_test, df_y_train, df_y_test = df_diabetic_loader.get_train_test_split()

In [8]:
df_diabetic_no_outliers_standardized

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,max_glu_serum_measured,A1Cresult_measured,weight_recorded
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,-1.136078,...,No,No,No,No,No,No,NO,0,0,0
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,-0.459430,...,No,No,No,No,Ch,Yes,>30,0,0,0
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,-0.797754,...,No,No,No,No,No,Yes,NO,0,0,0
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,-0.797754,...,No,No,No,No,Ch,Yes,NO,0,0,0
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,-1.136078,...,No,No,No,No,Ch,Yes,NO,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),NaN,1,3,7,-0.459430,...,No,No,No,No,Ch,Yes,>30,0,1,0
101762,443847782,74694222,AfricanAmerican,Female,[80-90),NaN,1,4,5,0.217217,...,No,No,No,No,No,Yes,NO,0,0,0
101763,443854148,41088789,Caucasian,Male,[70-80),NaN,1,1,7,-1.136078,...,No,No,No,No,Ch,Yes,NO,0,0,0
101764,443857166,31693671,Caucasian,Female,[80-90),NaN,2,3,7,1.908836,...,No,No,No,No,Ch,Yes,NO,0,0,0
